# QDSV/QIntent IBM Hardware-Oriented Syndrome Validation

This notebook prepares the first hardware-oriented validation step for the QDSV/QIntent LDPC/qLDPC-style post-decoding paper.

Goal:

- build a small parity-check syndrome extraction workflow;
- run it first on Aer;
- optionally run it on IBM Quantum hardware;
- convert syndrome counts into correction candidates;
- compare a confidence/min-weight baseline against QDSV/QIntent structured reranking;
- save reproducible evidence without exposing the private QDSV decision formula.

This is not a production qLDPC decoder and does not claim quantum advantage. It validates the handoff:

```text
hardware/simulator counts -> syndrome evidence -> candidate corrections -> QDSV/QIntent ranking -> audit evidence
```


## 1. Install Dependencies

QIntent does not require a QDSV API key for this public workflow. IBM hardware execution requires your IBM Quantum token, but the token is never saved in this notebook.


In [ ]:
!pip install -q --upgrade "git+https://github.com/qdsvquantum-afk/qintent.git"
!pip install -q --upgrade qiskit qiskit-aer qiskit-ibm-runtime pandas


## 2. Imports and Configuration

By default, this notebook runs only on Aer. Set `RUN_IBM_HARDWARE = True` only after the Aer run succeeds and after checking IBM queue status.


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

from qintent import QIntentClient

API_URL = "https://qdsv-api-568788785178.us-central1.run.app/api"
client = QIntentClient(api_url=API_URL, timeout=90)

SHOTS = 1024
RUN_IBM_HARDWARE = False
IBM_BACKEND_NAME = None  # Example: "ibm_brisbane". Leave None to use least_busy().

public_limits = client.spec().get("public_limits", {})
public_limits


## 3. Small Syndrome Extraction Circuit

We use a 3-data-qubit parity-check toy workflow with two syndrome bits:

```text
s0 = d0 xor d1
s1 = d1 xor d2
```

Qiskit displays classical bitstrings in most-significant-bit-first order, so the displayed syndrome is `s1s0`.


In [ ]:
def displayed_syndrome(error_qubits):
    err = set(error_qubits)
    s0 = int((0 in err) ^ (1 in err))
    s1 = int((1 in err) ^ (2 in err))
    return f"{s1}{s0}"


def build_syndrome_circuit(error_qubits=(), name=None):
    data = QuantumRegister(3, "d")
    anc = QuantumRegister(2, "a")
    syn = ClassicalRegister(2, "syn")
    qc = QuantumCircuit(data, anc, syn, name=name or "syndrome")

    for q in error_qubits:
        qc.x(data[q])

    qc.cx(data[0], anc[0])
    qc.cx(data[1], anc[0])
    qc.cx(data[1], anc[1])
    qc.cx(data[2], anc[1])
    qc.measure(anc[0], syn[0])
    qc.measure(anc[1], syn[1])
    return qc

SCENARIOS = [
    {"scenario_id": 0, "true_error": tuple(), "label": "no_error"},
    {"scenario_id": 1, "true_error": (0,), "label": "x0"},
    {"scenario_id": 2, "true_error": (1,), "label": "x1"},
    {"scenario_id": 3, "true_error": (2,), "label": "x2"},
]

for row in SCENARIOS:
    row["expected_syndrome"] = displayed_syndrome(row["true_error"])

circuits = [build_syndrome_circuit(row["true_error"], name=row["label"]) for row in SCENARIOS]
pd.DataFrame(SCENARIOS)


## 4. Run on Aer First

This cell is the required pre-hardware validation. Do not run IBM hardware until this produces sensible syndrome counts.


In [ ]:
simulator = AerSimulator()
compiled = transpile(circuits, simulator, optimization_level=1)

aer_counts = []
for circuit in compiled:
    result = simulator.run(circuit, shots=SHOTS).result()
    aer_counts.append(dict(result.get_counts()))

for row, counts in zip(SCENARIOS, aer_counts):
    row["aer_counts"] = counts
    row["aer_observed_syndrome"] = max(counts, key=counts.get)
    row["aer_observed_probability"] = counts[row["aer_observed_syndrome"]] / SHOTS

pd.DataFrame([
    {
        "scenario_id": row["scenario_id"],
        "label": row["label"],
        "true_error": " ".join(map(str, row["true_error"])) if row["true_error"] else "none",
        "expected_syndrome": row["expected_syndrome"],
        "observed_syndrome": row["aer_observed_syndrome"],
        "observed_probability": row["aer_observed_probability"],
        "counts": row["aer_counts"],
    }
    for row in SCENARIOS
])


## 5. Optional IBM Hardware Run

Only set `RUN_IBM_HARDWARE = True` after Aer works.

Token handling options:

- Colab Secrets: create `IBM_QUANTUM_TOKEN`;
- environment variable: `IBM_QUANTUM_TOKEN`;
- temporary prompt via `getpass()`.

The token is not written to files. The notebook uses Qiskit Runtime V2 Sampler for current IBM hardware workflows.


In [ ]:
ibm_counts = None
ibm_backend_name = None
ibm_job_id = None

if RUN_IBM_HARDWARE:
    try:
        from google.colab import userdata
        IBM_TOKEN = userdata.get("IBM_QUANTUM_TOKEN")
    except Exception:
        IBM_TOKEN = os.getenv("IBM_QUANTUM_TOKEN")

    if not IBM_TOKEN:
        IBM_TOKEN = getpass("IBM Quantum token: ")

    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
    backend = service.backend(IBM_BACKEND_NAME) if IBM_BACKEND_NAME else service.least_busy(
        operational=True,
        simulator=False,
        min_num_qubits=5,
    )
    ibm_backend_name = backend.name
    print("Selected backend:", ibm_backend_name)

    pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuits = [pass_manager.run(circuit) for circuit in circuits]

    sampler = Sampler(mode=backend)
    job = sampler.run(isa_circuits, shots=SHOTS)
    ibm_job_id = job.job_id()
    print("IBM job id:", ibm_job_id)

    result = job.result()
    ibm_counts = []
    for pub_result in result:
        ibm_counts.append(dict(pub_result.data.syn.get_counts()))

    for row, counts in zip(SCENARIOS, ibm_counts):
        row["ibm_counts"] = counts
        row["ibm_observed_syndrome"] = max(counts, key=counts.get)
        row["ibm_observed_probability"] = counts[row["ibm_observed_syndrome"]] / SHOTS

    display(pd.DataFrame([
        {
            "scenario_id": row["scenario_id"],
            "label": row["label"],
            "expected_syndrome": row["expected_syndrome"],
            "ibm_observed_syndrome": row["ibm_observed_syndrome"],
            "ibm_observed_probability": row["ibm_observed_probability"],
            "ibm_counts": row["ibm_counts"],
        }
        for row in SCENARIOS
    ]))
else:
    print("IBM hardware disabled. Set RUN_IBM_HARDWARE = True after Aer validation succeeds.")


## 6. Convert Counts into Candidate Correction Evidence

The candidate set includes no-error, single-error and two-error hypotheses. The observed syndrome comes from IBM hardware if enabled, otherwise from Aer.


In [ ]:
CANDIDATES = [tuple(), (0,), (1,), (2,), (0, 1), (0, 2), (1, 2)]
LOGICAL_SENSITIVE = {1}


def candidate_features(scenario, source="aer"):
    observed = scenario[f"{source}_observed_syndrome"]
    counts = scenario[f"{source}_counts"]
    observed_prob = counts.get(observed, 0) / SHOTS
    rows = []
    for idx, candidate in enumerate(CANDIDATES):
        predicted = displayed_syndrome(candidate)
        syndrome_match = int(predicted == observed)
        candidate_weight = len(candidate)
        sensitive_hits = sum(1 for q in candidate if q in LOGICAL_SENSITIVE)
        exact = set(candidate) == set(scenario["true_error"])

        syndrome_support = int((1000 if syndrome_match else 200) * observed_prob)
        check_consistency = 1000 if syndrome_match else 250
        decoder_confidence = max(250, int((1000 - 140 * candidate_weight) * observed_prob)) if syndrome_match else 200
        propagation_safety = max(100, 940 - 90 * candidate_weight)
        logical_preservation = max(100, 950 - 260 * sensitive_hits - 45 * candidate_weight)
        distance_safety = max(100, 930 - 210 * sensitive_hits - 40 * candidate_weight)
        syndrome_risk = min(900, 35 + 80 * (1 - syndrome_match) + 20 * candidate_weight)
        logical_risk = min(900, 45 + 240 * sensitive_hits + 25 * candidate_weight)
        entropy_adjustment = 18 if syndrome_match and candidate_weight > 0 else -15

        rows.append({
            "scenario_id": scenario["scenario_id"],
            "label": scenario["label"],
            "candidate_index": idx,
            "correction_qubits": " ".join(map(str, candidate)) if candidate else "none",
            "candidate_weight": candidate_weight,
            "observed_syndrome": observed,
            "predicted_syndrome": predicted,
            "syndrome_support": syndrome_support,
            "check_consistency": check_consistency,
            "logical_preservation": logical_preservation,
            "distance_safety": distance_safety,
            "decoder_confidence": decoder_confidence,
            "propagation_safety": propagation_safety,
            "syndrome_risk": syndrome_risk,
            "logical_risk": logical_risk,
            "syndrome_entropy_adjustment": entropy_adjustment,
            "exact_correction": exact,
            "logical_failure_proxy": bool(sensitive_hits and not exact),
        })
    return rows

SOURCE = "ibm" if RUN_IBM_HARDWARE and ibm_counts else "aer"
all_rows = []
for scenario in SCENARIOS:
    all_rows.extend(candidate_features(scenario, source=SOURCE))

candidate_df = pd.DataFrame(all_rows)
candidate_df.head(12)


## 7. QDSV/QIntent Structured Reranking

The public QIntent declaration exposes blocks and signals, not the private scoring formula.


In [ ]:
QINTENT_SOURCE = '''
find_rows("candidate_index")
  .using_structured_semantic_score([
      block("syndrome", [
          signal("syndrome_support", influence=30, priority=2),
          signal("check_consistency", influence=20, priority=1),
      ], influence=30, priority=2, risk_adjustment="syndrome_risk", adjustments=[
          adjustment("syndrome_entropy_adjustment", influence=5),
      ]),
      block("logical_safety", [
          signal("logical_preservation", influence=40, priority=3),
          signal("distance_safety", influence=20, priority=2),
      ], influence=40, priority=3, risk_adjustment="logical_risk"),
      block("decoder", [
          signal("decoder_confidence", influence=25, priority=1),
          signal("propagation_safety", influence=15, priority=2),
      ], influence=30, priority=1),
  ], global_risk="logical_risk", profile="qldpc_ibm_hardware_syndrome_validation")
  .accept_if(threshold=600)
  .rank()
  .top_k(1)
'''

print(QINTENT_SOURCE)
explain = client.explain(QINTENT_SOURCE, rows=all_rows)
explain.get("operation") or explain.get("compiled") or explain


In [ ]:
summary_rows = []
selected_rows = []

for scenario in SCENARIOS:
    rows = [row for row in all_rows if row["scenario_id"] == scenario["scenario_id"]]
    qdsv_result = client.run(QINTENT_SOURCE, rows=rows)
    qdsv_rows = qdsv_result.get("result", {}).get("selected_rows", [])
    qdsv_top = qdsv_rows[0] if qdsv_rows else None

    baseline_pool = sorted(
        rows,
        key=lambda r: (-r["decoder_confidence"], r["candidate_weight"], r["logical_risk"]),
    )
    baseline = baseline_pool[0]

    if qdsv_top:
        original = next(r for r in rows if r["candidate_index"] == qdsv_top["candidate_index"])
        selected_rows.append({**original, **{k: v for k, v in qdsv_top.items() if k.startswith("_qdsv")}})
    else:
        original = None

    summary_rows.append({
        "scenario_id": scenario["scenario_id"],
        "label": scenario["label"],
        "source": SOURCE,
        "expected_syndrome": scenario["expected_syndrome"],
        "observed_syndrome": scenario[f"{SOURCE}_observed_syndrome"],
        "baseline_qubits": baseline["correction_qubits"],
        "baseline_confidence": baseline["decoder_confidence"],
        "baseline_logical_risk": baseline["logical_risk"],
        "baseline_exact": baseline["exact_correction"],
        "baseline_failure_proxy": baseline["logical_failure_proxy"],
        "qdsv_qubits": original["correction_qubits"] if original else None,
        "qdsv_confidence": original["decoder_confidence"] if original else None,
        "qdsv_logical_risk": original["logical_risk"] if original else None,
        "qdsv_exact": original["exact_correction"] if original else None,
        "qdsv_failure_proxy": original["logical_failure_proxy"] if original else None,
        "risk_delta": baseline["logical_risk"] - (original["logical_risk"] if original else baseline["logical_risk"]),
    })

summary_df = pd.DataFrame(summary_rows)
selected_df = pd.DataFrame(selected_rows)
summary_df


## 8. Save Evidence

These files are the artifacts to archive back into the repository after the run.


In [ ]:
evidence = {
    "experiment": "qldpc_ibm_hardware_syndrome_validation",
    "source": SOURCE,
    "shots": SHOTS,
    "run_ibm_hardware": RUN_IBM_HARDWARE,
    "ibm_backend_name": ibm_backend_name,
    "ibm_job_id": ibm_job_id,
    "scenarios": SCENARIOS,
    "qintent_source": QINTENT_SOURCE,
    "summary": summary_rows,
    "selected_rows": selected_rows,
    "notes": [
        "This validates the handoff from simulator/hardware syndrome counts to QDSV/QIntent post-decoding candidate ranking.",
        "This is a small parity-check hardware-oriented validation, not a production qLDPC decoder.",
        "The private QDSV scoring formula is not exposed in the public evidence.",
    ],
}

summary_path = Path("qdsv_qldpc_ibm_hardware_syndrome_summary.csv")
evidence_path = Path("qdsv_qldpc_ibm_hardware_syndrome_evidence.json")

summary_df.to_csv(summary_path, index=False)
evidence_path.write_text(json.dumps(evidence, indent=2, default=str), encoding="utf-8")

print("Saved:", summary_path)
print("Saved:", evidence_path)
